# ML-Driven Volatility Impact Engine

This notebook is the first working prototype for a **data-driven news impact engine** that can later feed an MQL5 EA.

It now uses the **existing processed workspace datasets** instead of placeholder loaders, and it is configured for a **Python 3.12 + ROCm PyTorch** environment.

## Current objective

Build a model that estimates whether a calendar event is likely to create dangerous post-news trading conditions, then export a practical CSV for EA-side filtering and future policy logic.

## Confirmed design choices

- Economic calendar ingestion: **from scratch upstream, but notebook consumes processed parquet outputs**
- Massive.com price source: **S3 flat files**, updated monthly
- Market granularity: **1-minute bars**
- Modeling target: **both score + bucket/class**
- Primary use case: **identify post-news sessions to avoid trading**
- Immediate export goal: CSV showing **Date / Forex Symbol / Score / Bucket Class**

## Section 1 - Environment and paths

This section confirms the runtime environment, including ROCm-enabled PyTorch, and defines the key dataset paths used by the notebook.

In [ ]:
from __future__ import annotations

import math
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path('/home/asurani/.openclaw/workspace/projects/volatility-impact-engine')
WORKSPACE_ROOT = Path('/home/asurani/.openclaw/workspace')

CALENDAR_EVENTS_PATH = WORKSPACE_ROOT / 'data' / 'processed' / 'economic_calendar' / 'events.parquet'
JOINED_CONTEXT_PATH = WORKSPACE_ROOT / 'data' / 'processed' / 'joined' / 'event_market_context_quantile_labeled.parquet'
TRAINING_DATASET_PATH = WORKSPACE_ROOT / 'data' / 'processed' / 'modeling' / 'event_risk_training_dataset.parquet'
AVOID_SESSION_DATASET_PATH = WORKSPACE_ROOT / 'data' / 'processed' / 'modeling' / 'avoid_session_training_dataset.parquet'
MASSIVE_MINUTE_ROOT = WORKSPACE_ROOT / 'data' / 'processed' / 'massive' / 'fx_minute_bars'
EXPORT_DIR = PROJECT_ROOT / 'data'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch version:', torch.__version__)
print('ROCm/HIP version:', getattr(torch.version, 'hip', None))
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Section 2 - Load existing processed datasets

The upstream engineering work already produced clean parquet datasets.

That means this notebook can focus on modeling instead of rebuilding the full ingestion stack. We load the processed calendar events, joined event/market context, and the event risk modeling table.

In [ ]:
calendar_df = pd.read_parquet(CALENDAR_EVENTS_PATH)
joined_df = pd.read_parquet(JOINED_CONTEXT_PATH)
training_df = pd.read_parquet(TRAINING_DATASET_PATH)

calendar_df['event_timestamp_utc'] = pd.to_datetime(calendar_df['event_timestamp_utc'], utc=True)
joined_df['event_timestamp_utc'] = pd.to_datetime(joined_df['event_timestamp_utc'], utc=True)
training_df['event_timestamp_utc'] = pd.to_datetime(training_df['event_timestamp_utc'], utc=True)

print('calendar_df:', calendar_df.shape)
print('joined_df:', joined_df.shape)
print('training_df:', training_df.shape)
print('pairs:', sorted(training_df['pair'].unique().tolist()))

## Section 3 - Understand targets and business framing

You asked for both a **continuous score** and a **bucketed class**, with the practical goal of deciding which news windows or sessions should be avoided.

The current processed data already includes several useful target definitions:

- `y_volatility_expansion`
- `y_trend_danger`
- `y_whipsaw_danger`
- `y_post_news_session_risk`

For the first practical model, we use **post-news session risk** as the main binary signal, then convert the predicted probability into a human-friendly score and class.

In [ ]:
target_cols = [
    'y_trend_danger',
    'y_whipsaw_danger',
    'y_volatility_expansion',
    'y_post_news_session_risk',
]

for col in target_cols:
    print(f'\n{col}')
    print(training_df[col].value_counts(dropna=False))

## Section 4 - Feature set definition

The processed modeling dataset already contains a compact set of event and pair features.

We keep the first model deliberately simple and robust:
- calendar importance
- event timing features
- pair identity features
- currency indicators
- event category indicators
- upstream risk priors and avoid-session hints

This gives us a strong baseline before introducing sequence models or raw minute-bar encoders.

In [ ]:
META_COLS = ['event_id', 'pair', 'event_timestamp_utc', 'event_name', 'event_category', 'currency_norm', 'risk_prior', 'avoid_session']
TARGET_COL = 'y_post_news_session_risk'

feature_cols = [
    c for c in training_df.columns
    if c not in META_COLS + target_cols
]

feature_frame = training_df[feature_cols].copy()
target_series = training_df[TARGET_COL].astype(float).copy()

print('feature count:', len(feature_cols))
print(feature_cols)

## Section 5 - Train/validation split

For this first prototype we use a random split.

For a production version, we should switch to a **time-aware split** so future events are never predicted using future information leaked through the training period.

In [ ]:
X = feature_frame.values.astype(np.float32)
y = target_series.values.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_scaled,
    y,
    training_df.index.values,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('train size:', X_train.shape, y_train.shape)
print('val size:', X_val.shape, y_val.shape)

## Section 6 - PyTorch dataset and classifier

We model post-news session risk as a binary classification task. The model outputs a probability, which later becomes both:

- a continuous **score** in `[0, 1]`
- a bucketed **class** such as `low`, `medium`, `high`, or `extreme`

That keeps the output simple for CSV export and EA-side rule wiring.

In [ ]:
class BinaryRiskDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class RiskMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

## Section 7 - Model training

This section trains the classifier and reports validation quality.

Because the target is somewhat imbalanced, we use `BCEWithLogitsLoss` with a positive-class weight.

In [ ]:
train_ds = BinaryRiskDataset(X_train, y_train)
val_ds = BinaryRiskDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

pos_weight_value = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1.0)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)

model = RiskMLP(input_dim=X_train.shape[1], hidden_dim=128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

history = []
EPOCHS = 12

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    val_probs = []
    val_true = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.sigmoid(logits)
            val_losses.append(loss.item())
            val_probs.extend(probs.cpu().numpy().ravel().tolist())
            val_true.extend(yb.cpu().numpy().ravel().tolist())

    val_pred = (np.array(val_probs) >= 0.5).astype(int)
    val_true_arr = np.array(val_true).astype(int)
    auc = roc_auc_score(val_true_arr, val_probs)

    history.append({
        'epoch': epoch + 1,
        'train_loss': float(np.mean(train_losses)),
        'val_loss': float(np.mean(val_losses)),
        'val_auc': float(auc),
        'val_positive_rate': float(val_pred.mean()),
    })

history_df = pd.DataFrame(history)
history_df

## Section 8 - Validation diagnostics

This section checks model behavior on the validation set.

For EA integration, the most important thing is not academic perfection — it is whether the model separates low-risk from dangerous post-news conditions well enough to help decide when to trade and when to skip.

In [ ]:
display(history_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=history_df, x='epoch', y='train_loss', ax=axes[0], label='train')
sns.lineplot(data=history_df, x='epoch', y='val_loss', ax=axes[0], label='val')
axes[0].set_title('Loss by Epoch')

sns.lineplot(data=history_df, x='epoch', y='val_auc', ax=axes[1], label='AUC')
axes[1].set_title('Validation AUC by Epoch')
plt.tight_layout()
plt.show()

val_probs_np = np.array(val_probs)
val_pred_np = (val_probs_np >= 0.5).astype(int)
val_true_np = np.array(val_true).astype(int)

print(classification_report(val_true_np, val_pred_np, digits=4))
print('Confusion matrix:')
print(confusion_matrix(val_true_np, val_pred_np))

## Section 9 - Generate score and bucket outputs

We now score the full dataset.

The probability output becomes the **continuous score**. Then we map that score into an operational bucket:

- `low`
- `medium`
- `high`
- `extreme`

These thresholds are intentionally simple for the first pass and can later be optimized using business costs or EA backtest outcomes.

In [ ]:
def bucketize_score(score: float) -> str:
    if score < 0.25:
        return 'low'
    if score < 0.50:
        return 'medium'
    if score < 0.75:
        return 'high'
    return 'extreme'

model.eval()
with torch.no_grad():
    full_logits = model(torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE))
    full_scores = torch.sigmoid(full_logits).cpu().numpy().ravel()

scored_df = training_df[['event_timestamp_utc', 'pair', 'event_name', 'currency_norm']].copy()
scored_df['score'] = full_scores
scored_df['bucket_class'] = scored_df['score'].apply(bucketize_score)
scored_df['score_pct'] = (100.0 * scored_df['score']).round(2)

scored_df.head()

## Section 10 - Export the CSV formats requested

You asked for a CSV that can immediately be inspected and later consumed by the EA.

This notebook exports two practical views:

1. **Date / Currency / Calendar Event / Score / Class**
2. **Date / Forex Symbol / Score / Bucket Class**

Both are useful:
- the first is easier for review
- the second is closer to EA execution logic

In [ ]:
event_review_export = scored_df.rename(columns={
    'event_timestamp_utc': 'Date',
    'currency_norm': 'Currency',
    'event_name': 'Calendar Event',
    'score_pct': 'Score',
    'bucket_class': 'Class',
})[['Date', 'Currency', 'Calendar Event', 'Score', 'Class']]

symbol_export = scored_df.rename(columns={
    'event_timestamp_utc': 'Date',
    'pair': 'Forex Symbol',
    'score_pct': 'Score',
    'bucket_class': 'Bucket Class',
})[['Date', 'Forex Symbol', 'Score', 'Bucket Class']]

event_review_export_path = EXPORT_DIR / 'event_impact_review.csv'
symbol_export_path = EXPORT_DIR / 'event_impact_symbol_scores.csv'

event_review_export.to_csv(event_review_export_path, index=False)
symbol_export.to_csv(symbol_export_path, index=False)

print('Wrote:', event_review_export_path)
print('Wrote:', symbol_export_path)
display(event_review_export.head(10))
display(symbol_export.head(10))

## Section 11 - Optional future upgrades

This first version is already useful, but the obvious next upgrades are:

1. Replace random split with **time-based validation**
2. Train separate models per pair or per session
3. Blend binary danger targets into a single composite score
4. Add direct features from raw minute-bar windows around the event
5. Export a rolling daily or weekly forecast file for live EA use
6. Add MQL5-side threshold rules such as `skip trade if score >= 70 and class in {high, extreme}`
7. Retrain monthly when new Massive S3 flat files land